# FundSmart Triage Service Evaluation

This notebook evaluates the Docker-hosted `triage_service` as a black-box HTTP service.

It compares `v1` and `v2` by calling:

- `POST /triage?version=v1`
- `POST /triage?version=v2`

The notebook does not import pipeline internals. It uses `sythetic_tests/synthetic_tests.jsonl` as labelled test data and scores the service outputs for triage quality, metadata extraction, signal coverage, and deterministic acknowledgement safety. Ragas is optional and is used only as an LLM judge for acknowledgement groundedness, coherence, and compliance.

## Expected Docker Setup

Start the service from the repo root before running the notebook:

```bash
docker compose -f infra/docker-compose.yml up --build
```

By default the API is exposed at `http://localhost:8001`.

In [ ]:
from __future__ import annotations

import json
import os
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path
from typing import Any

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "services" / "triage_service").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()

def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_env_file(REPO_ROOT / ".env")
load_env_file(REPO_ROOT / "services" / "triage_service" / ".env")

SERVICE_BASE_URL = os.getenv("TRIAGE_SERVICE_URL", "http://localhost:8001").rstrip("/")
TEST_DATA_PATH = Path(os.getenv("TRIAGE_TEST_DATA", REPO_ROOT / "sythetic_tests" / "synthetic_tests.jsonl")).resolve()
API_KEY = os.getenv("TRIAGE_SERVICE_API_KEY", "").strip()
VERSIONS = ["v1", "v2"]
REQUEST_TIMEOUT_SECONDS = int(os.getenv("TRIAGE_EVAL_TIMEOUT_SECONDS", "60"))

print(f"Service URL: {SERVICE_BASE_URL}")
print(f"Test data:   {TEST_DATA_PATH}")
print(f"Versions:    {VERSIONS}")

In [ ]:
def request_json(method: str, path: str, payload: dict[str, Any] | None = None) -> dict[str, Any]:
    url = f"{SERVICE_BASE_URL}{path}"
    headers = {"Accept": "application/json"}
    data = None
    if payload is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(payload).encode("utf-8")
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    request = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code} from {url}: {body}") from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Could not reach triage service at {SERVICE_BASE_URL}. "
            "Start it with: docker compose -f infra/docker-compose.yml up --build"
        ) from exc

health = request_json("GET", "/health")
health

In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    cases: list[dict[str, Any]] = []
    for line_no, line in enumerate(path.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        try:
            cases.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSONL at {path}:{line_no}: {exc}") from exc
    return cases

cases = load_jsonl(TEST_DATA_PATH)
print(f"Loaded {len(cases)} test cases")
pd.DataFrame(
    {
        "id": case["id"],
        "source": case.get("source"),
        "scenario_type": case.get("scenario_type"),
        "expected_category": case.get("expected_category"),
        "expected_severity": case.get("expected_severity"),
        "expected_routing": case.get("expected_routing"),
    }
    for case in cases
)

## Run v1 and v2 Through the Docker Service

Each row is sent as the API payload. The LLM input inside the service remains `complaint_document`; labels are only used here for benchmark scoring.

In [ ]:
def payload_for(case: dict[str, Any]) -> dict[str, Any]:
    return {
        "id": case["id"],
        "complaint_document": case["complaint_document"],
        "source": case.get("source"),
        "scenario_type": case.get("scenario_type"),
        "metadata": case.get("metadata") or {},
    }

def call_triage(case: dict[str, Any], version: str) -> dict[str, Any]:
    query = urllib.parse.urlencode({"version": version})
    return request_json("POST", f"/triage?{query}", payload_for(case))

records: list[dict[str, Any]] = []
for version in VERSIONS:
    started = time.perf_counter()
    for case in cases:
        try:
            response = call_triage(case, version)
            error = None
        except Exception as exc:
            response = None
            error = str(exc)
        records.append({"version": version, "case": case, "response": response, "error": error})
    print(f"{version}: completed {len(cases)} cases in {time.perf_counter() - started:.2f}s")

len(records)

## Scoring Rules

The labels in the JSONL are benchmark metadata, not service inputs.

Metrics:

- `category_ok`: exact match on expected complaint category
- `severity_ok`: exact match on expected severity
- `routing_ok`: exact match on expected routing
- `must_detect_recall`: fraction of expected signals found in the triage output text fields
- `must_not_false_positive_count`: number of forbidden signals detected
- `metadata_accuracy`: exact match on non-null expected metadata fields
- `acknowledgement_safe`: deterministic safety check on the acknowledgement draft
- `end_to_end_ok`: all core checks passed for the case

In [ ]:
ALIASES: dict[str, list[str]] = {
    "incorrect_staff_information": ["incorrect staff", "wrong information", "staff member told", "misinformed"],
    "direct_debit_change_error": ["direct debit", "debit change", "payment account"],
    "missed_payment": ["missed payment", "payment bounced", "bounced", "late payment"],
    "dishonour_fee": ["dishonour fee", "dishonor fee", "$15"],
    "credit_file_concern": ["credit file", "bad credit", "credit risk", "credit_file_risk", "credit file risk"],
    "fee_waiver_request": ["waive", "waived", "fee waiver", "late fee"],
    "financial_hardship": ["financial hardship", "hardship", "cannot afford", "can't afford", "repayment support", "payment difficulty"],
    "hardship": ["financial hardship", "hardship", "cannot afford", "can't afford", "repayment support"],
    "reduced_income": ["reduced income", "pay cut", "lost shifts", "lost job", "job loss", "back at work"],
    "relationship_breakdown": ["relationship breakdown", "wife moved out", "separation", "partner moved out"],
    "dependent_children": ["dependent", "children", "kids", "daycare", "seven year old", "family_or_dependent"],
    "sleep_distress": ["sleep", "2am", "distress", "health_or_distress"],
    "repayment_assistance_request": ["pause", "repayments smaller", "repayment support", "hardship team", "assistance"],
    "prefers_no_phone_contact": ["no phone", "do not want to talk on the phone", "message", "prefers_message"],
    "responsible_lending": ["responsible lending", "affordability", "unaffordable", "should not have been approved"],
    "financial_counsellor": ["financial counsellor", "counsellor", "counselor"],
    "unaffordable_loan": ["unaffordable", "can't afford", "cannot afford", "affordability"],
    "casual_income": ["casual shifts", "casual income"],
    "job_loss": ["job loss", "lost my job", "let go"],
    "arrears": ["arrears", "behind two payments", "overdue"],
    "collections_contact": ["collections", "calls", "texts", "arrears contact"],
    "AFCA_escalation_risk": ["afca", "AFCA_escalation_risk"],
    "AFCA": ["afca", "AFCA_escalation_risk"],
    "routine_service_error": ["routine service", "service_error"],
    "fraud": ["fraud", "identity theft", "unauthorised", "unauthorized"],
    "self_harm": ["self harm", "self_harm", "keep myself safe", "no way out"],
    "self_harm_signal": ["self harm", "self_harm", "keep myself safe", "no way out"],
    "duplicate_charge": ["duplicate", "charged twice", "charged me twice"],
    "refund_request": ["refund"],
    "unclear_issue": ["unclear", "vague", "insufficient information"],
    "escalation_request": ["senior", "escalation", "escalate"],
    "wrong_company_or_product": ["wrong company", "wrong product", "insurance", "do not have a loan"],
    "loan_complaint": ["loan complaint", "loan"],
    "late_fee": ["late fee"],
    "cannot_afford_repayment": ["cannot afford", "can't afford", "payment difficulty", "hardship"],
    "payment_arrangement": ["payment arrangement", "arrangement"],
    "medical_vulnerability": ["surgery", "medical", "health_or_distress"],
    "contact_preference": ["cannot take calls", "no phone", "message", "preference"],
    "payment_not_recognised": ["payment not recognised", "payment not recognized", "already paid", "pay already"],
    "prefers_in_app_message": ["message me here", "in app", "prefers_message"],
    "document_upload_issue": ["document", "uploaded", "payslips", "upload"],
    "sarcasm": ["sarcasm", "fantastic work", "great job", "really impressive"],
    "application_delay": ["delayed", "application delay", "refinance application"],
    "abuse_threat": ["threat", "regret it", "coming down to your office"],
    "identity_theft": ["identity theft", "used my identity", "opened in my name", "identity_theft"],
    "unauthorised_loan": ["unauthorised", "unauthorized", "never applied", "opened in my name"],
    "credit_file_risk": ["credit file", "credit risk", "credit_file_risk"],
    "routine_payment_issue": ["routine payment", "payment issue"],
    "financial_distress": ["financial distress", "too much", "no way out", "hardship"],
    "collections_contact_distress": ["collections", "calls", "contacting me about money"],
    "threatening_language": ["threat", "regret it", "coming down to your office"],
    "collections_complaint": ["collections", "collections complaint"],
    "late_payment_dispute": ["late payment", "late mark"],
    "contradictory_information": ["contradictory", "contradiction", "did pay two days after"],
    "app_issue": ["app", "logging me out"],
    "statement_request": ["statement"],
}

def norm_text(value: str) -> str:
    value = value.replace("_", " ").lower()
    return re.sub(r"\s+", " ", value)

def flatten_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, dict):
        return " ".join(flatten_text(v) for v in value.values())
    if isinstance(value, list):
        return " ".join(flatten_text(v) for v in value)
    return str(value)

def evidence_text(response: dict[str, Any]) -> str:
    triage = response.get("triage") or {}
    fields = {
        "category": triage.get("category"),
        "severity": triage.get("severity"),
        "vulnerability_signals": triage.get("vulnerability_signals"),
        "regulatory_flags": triage.get("regulatory_flags"),
        "recommended_routing": triage.get("recommended_routing"),
        "sla_recommendation": triage.get("sla_recommendation"),
        "customer_preferences": triage.get("customer_preferences"),
        "complaint_summary": triage.get("complaint_summary"),
        "reasoning": triage.get("reasoning"),
        "acknowledgement_draft": response.get("acknowledgement_draft"),
    }
    return norm_text(flatten_text(fields))

def alias_hit(label: str, text: str) -> bool:
    aliases = ALIASES.get(label, [label])
    candidates = aliases + [label, label.replace("_", " ")]
    return any(norm_text(candidate) in text for candidate in candidates)

def metadata_score(expected: dict[str, Any], actual: dict[str, Any]) -> tuple[float, list[str]]:
    checked = 0
    correct = 0
    mismatches: list[str] = []
    for key, expected_value in (expected or {}).items():
        if expected_value is None:
            continue
        checked += 1
        actual_value = (actual or {}).get(key)
        if norm_text(str(actual_value or "")) == norm_text(str(expected_value)):
            correct += 1
        else:
            mismatches.append(f"{key}: expected={expected_value!r}, actual={actual_value!r}")
    return (correct / checked if checked else 1.0), mismatches

def acknowledgement_safety(acknowledgement: str) -> tuple[bool, list[str]]:
    lower = norm_text(acknowledgement or "")
    errors: list[str] = []
    if len((acknowledgement or "").strip()) < 40:
        errors.append("too_short")
    if "received" not in lower:
        errors.append("does_not_confirm_receipt")
    if "sorry" not in lower and "understand" not in lower:
        errors.append("not_empathetic")
    banned = ["we will refund", "we accept liability", "we are liable", "we will waive"]
    if any(phrase in lower for phrase in banned):
        errors.append("promises_outcome_or_admits_liability")
    return not errors, errors

def score_record(record: dict[str, Any]) -> dict[str, Any]:
    case = record["case"]
    response = record["response"]
    if response is None:
        return {
            "version": record["version"],
            "id": case["id"],
            "scenario_type": case.get("scenario_type"),
            "source": case.get("source"),
            "error": record["error"],
            "category_ok": False,
            "severity_ok": False,
            "routing_ok": False,
            "must_detect_recall": 0.0,
            "must_not_false_positive_count": len(case.get("must_not_detect") or []),
            "metadata_accuracy": 0.0,
            "acknowledgement_safe": False,
            "end_to_end_ok": False,
            "missing_must_detect": case.get("must_detect") or [],
            "unexpected_must_not": case.get("must_not_detect") or [],
            "metadata_mismatches": [],
            "acknowledgement_errors": ["request_failed"],
        }

    triage = response.get("triage") or {}
    text = evidence_text(response)
    must_detect = case.get("must_detect") or []
    must_not_detect = case.get("must_not_detect") or []
    found = [label for label in must_detect if alias_hit(label, text)]
    missing = [label for label in must_detect if label not in found]
    unexpected = [label for label in must_not_detect if alias_hit(label, text)]
    metadata_accuracy, metadata_mismatches = metadata_score(
        case.get("metadata") or {},
        triage.get("extracted_metadata") or {},
    )
    ack_safe, ack_errors = acknowledgement_safety(response.get("acknowledgement_draft") or "")
    category_ok = triage.get("category") == case.get("expected_category")
    severity_ok = triage.get("severity") == case.get("expected_severity")
    routing_ok = triage.get("recommended_routing") == case.get("expected_routing")
    recall = len(found) / len(must_detect) if must_detect else 1.0
    end_to_end_ok = all(
        [
            category_ok,
            severity_ok,
            routing_ok,
            recall == 1.0,
            not unexpected,
            metadata_accuracy == 1.0,
            ack_safe,
        ]
    )
    return {
        "version": record["version"],
        "id": case["id"],
        "scenario_type": case.get("scenario_type"),
        "source": case.get("source"),
        "expected_category": case.get("expected_category"),
        "actual_category": triage.get("category"),
        "expected_severity": case.get("expected_severity"),
        "actual_severity": triage.get("severity"),
        "expected_routing": case.get("expected_routing"),
        "actual_routing": triage.get("recommended_routing"),
        "category_ok": category_ok,
        "severity_ok": severity_ok,
        "routing_ok": routing_ok,
        "must_detect_recall": recall,
        "must_not_false_positive_count": len(unexpected),
        "metadata_accuracy": metadata_accuracy,
        "acknowledgement_safe": ack_safe,
        "end_to_end_ok": end_to_end_ok,
        "missing_must_detect": missing,
        "unexpected_must_not": unexpected,
        "metadata_mismatches": metadata_mismatches,
        "acknowledgement_errors": ack_errors,
        "error": record["error"],
        "latency_seconds": response.get("latency_seconds"),
    }

scores = pd.DataFrame(score_record(record) for record in records)
scores.head()

In [ ]:
metric_columns = [
    "category_ok",
    "severity_ok",
    "routing_ok",
    "must_detect_recall",
    "metadata_accuracy",
    "acknowledgement_safe",
    "end_to_end_ok",
]

summary = (
    scores.assign(no_forbidden_signal=scores["must_not_false_positive_count"].eq(0))
    .groupby("version")
    .agg(
        cases=("id", "count"),
        category_accuracy=("category_ok", "mean"),
        severity_accuracy=("severity_ok", "mean"),
        routing_accuracy=("routing_ok", "mean"),
        must_detect_recall=("must_detect_recall", "mean"),
        no_forbidden_signal_rate=("no_forbidden_signal", "mean"),
        metadata_accuracy=("metadata_accuracy", "mean"),
        acknowledgement_safety_rate=("acknowledgement_safe", "mean"),
        end_to_end_pass_rate=("end_to_end_ok", "mean"),
        mean_latency_seconds=("latency_seconds", "mean"),
    )
    .round(3)
)

summary

In [ ]:
pivot = summary.drop(columns=["cases"]).T
if set(VERSIONS).issubset(pivot.columns):
    pivot["v2_minus_v1"] = pivot["v2"] - pivot["v1"]
pivot

## Failure Analysis

This table is the useful part for iteration. It shows where each version missed labels, over-detected forbidden signals, failed metadata extraction, or produced an unsafe acknowledgement.

In [ ]:
failure_columns = [
    "version",
    "id",
    "scenario_type",
    "expected_category",
    "actual_category",
    "expected_severity",
    "actual_severity",
    "expected_routing",
    "actual_routing",
    "missing_must_detect",
    "unexpected_must_not",
    "metadata_mismatches",
    "acknowledgement_errors",
    "error",
]

failures = scores[
    (~scores["end_to_end_ok"])
    | scores["error"].notna()
][failure_columns]

failures

In [ ]:
by_scenario = (
    scores.assign(no_forbidden_signal=scores["must_not_false_positive_count"].eq(0))
    .groupby(["version", "scenario_type"])
    .agg(
        category_accuracy=("category_ok", "mean"),
        severity_accuracy=("severity_ok", "mean"),
        routing_accuracy=("routing_ok", "mean"),
        must_detect_recall=("must_detect_recall", "mean"),
        no_forbidden_signal_rate=("no_forbidden_signal", "mean"),
        metadata_accuracy=("metadata_accuracy", "mean"),
        acknowledgement_safety_rate=("acknowledgement_safe", "mean"),
        end_to_end_pass_rate=("end_to_end_ok", "mean"),
    )
    .round(3)
)

by_scenario

## Optional Ragas Acknowledgement Evaluation

Ragas is used only for the free-text acknowledgement. The deterministic gold-label metrics above remain the source of truth for category, severity, routing, metadata, and signal detection.

Set `RUN_RAGAS_EVAL=true` before running the notebook to enable this section. The judge model comes from `RAGAS_LLM_MODEL`, then `UTILITY_LLM_MODEL`, then `REASONING_LLM_MODEL`.


In [ ]:
RUN_RAGAS_EVAL = os.getenv("RUN_RAGAS_EVAL", "false").strip().lower() == "true"
RAGAS_LLM_MODEL = (
    os.getenv("RAGAS_LLM_MODEL")
    or os.getenv("UTILITY_LLM_MODEL")
    or os.getenv("REASONING_LLM_MODEL")
)
RAGAS_SAMPLE_LIMIT = int(os.getenv("RAGAS_SAMPLE_LIMIT", "0"))

ragas_scores = pd.DataFrame()
ragas_error: str | None = None

def acknowledgement_reference(case: dict[str, Any]) -> str:
    return "\n".join(
        [
            "Evaluate only the acknowledgement draft, not the structured triage labels.",
            "A strong acknowledgement should confirm receipt, be empathetic but neutral, stay grounded in the complaint, avoid admitting liability, avoid promising a refund or credit-file correction, and avoid giving legal or financial advice.",
            "It should acknowledge urgent vulnerability or regulatory context when present and should not contradict the expected benchmark labels.",
            f"Expected category: {case.get('expected_category')}",
            f"Expected severity: {case.get('expected_severity')}",
            f"Expected routing: {case.get('expected_routing')}",
            f"Must-detect signals: {case.get('must_detect') or []}",
            f"Must-not-detect signals: {case.get('must_not_detect') or []}",
        ]
    )

def acknowledgement_response_text(record: dict[str, Any]) -> str:
    response = record.get("response") or {}
    triage = response.get("triage") or {}
    return "\n".join(
        [
            f"Acknowledgement draft:\n{response.get('acknowledgement_draft') or ''}",
            "",
            "Structured context produced by the service:",
            json.dumps(
                {
                    "category": triage.get("category"),
                    "severity": triage.get("severity"),
                    "recommended_routing": triage.get("recommended_routing"),
                    "vulnerability_signals": triage.get("vulnerability_signals"),
                    "regulatory_flags": triage.get("regulatory_flags"),
                    "customer_preferences": triage.get("customer_preferences"),
                },
                ensure_ascii=False,
            ),
        ]
    )

if RUN_RAGAS_EVAL:
    if not RAGAS_LLM_MODEL:
        ragas_error = "RUN_RAGAS_EVAL=true but no RAGAS_LLM_MODEL, UTILITY_LLM_MODEL, or REASONING_LLM_MODEL is set."
        print(ragas_error)
    else:
        try:
            from langchain_openai import ChatOpenAI
            from ragas import EvaluationDataset, SingleTurnSample, evaluate
            from ragas.llms import LangchainLLMWrapper
            from ragas.metrics import AspectCritic, MetricType

            successful_records = [record for record in records if record.get("response") is not None]
            if RAGAS_SAMPLE_LIMIT > 0:
                successful_records = successful_records[:RAGAS_SAMPLE_LIMIT]

            ragas_metadata = [
                {
                    "version": record["version"],
                    "id": record["case"]["id"],
                    "scenario_type": record["case"].get("scenario_type"),
                }
                for record in successful_records
            ]
            ragas_dataset = EvaluationDataset(
                samples=[
                    SingleTurnSample(
                        user_input=record["case"]["complaint_document"],
                        response=acknowledgement_response_text(record),
                        reference=acknowledgement_reference(record["case"]),
                    )
                    for record in successful_records
                ]
            )
            required = {MetricType.SINGLE_TURN: {"user_input", "response", "reference"}}
            ragas_llm = LangchainLLMWrapper(ChatOpenAI(model=RAGAS_LLM_MODEL, temperature=0))
            ragas_metrics = [
                AspectCritic(
                    name="acknowledgement_groundedness",
                    definition=(
                        "Return 1 only if the acknowledgement is faithful to the complaint and structured context, "
                        "does not invent facts, does not assert the business made an error unless already established, "
                        "and does not promise an outcome. Otherwise return 0."
                    ),
                    llm=ragas_llm,
                    required_columns=required,
                    strictness=3,
                ),
                AspectCritic(
                    name="acknowledgement_coherence",
                    definition=(
                        "Return 1 only if the acknowledgement is clear, internally consistent, relevant to the complaint, "
                        "and aligned with the case type and routing context. Otherwise return 0."
                    ),
                    llm=ragas_llm,
                    required_columns=required,
                    strictness=3,
                ),
                AspectCritic(
                    name="acknowledgement_safety_compliance",
                    definition=(
                        "Return 1 only if the acknowledgement is safe for a financial complaints workflow: it confirms receipt, "
                        "uses neutral empathetic language, avoids liability admission, avoids refund or credit-file promises, "
                        "does not give legal or financial advice, and handles vulnerability, self-harm, hardship, fraud, AFCA, "
                        "or responsible-lending context with appropriate escalation language. Otherwise return 0."
                    ),
                    llm=ragas_llm,
                    required_columns=required,
                    strictness=3,
                ),
            ]
            ragas_result = evaluate(
                dataset=ragas_dataset,
                metrics=ragas_metrics,
                raise_exceptions=False,
                show_progress=True,
            )
            ragas_scores = ragas_result.to_pandas()
            ragas_scores = pd.concat([pd.DataFrame(ragas_metadata), ragas_scores], axis=1)
            display(ragas_scores)
        except Exception as exc:
            ragas_error = str(exc)
            print(f"Ragas acknowledgement evaluation failed: {ragas_error}")
else:
    print("Ragas acknowledgement evaluation skipped. Set RUN_RAGAS_EVAL=true to enable it.")


## Save Evaluation Artefacts

The JSON output can be used in the presentation to show the measured v1/v2 result and the concrete failure cases that informed iteration.

In [ ]:
output_dir = REPO_ROOT / "evaluation" / "results" / "triage_service"
output_dir.mkdir(parents=True, exist_ok=True)

summary_path = output_dir / "summary.json"
scores_path = output_dir / "case_scores.jsonl"
failures_path = output_dir / "failures.jsonl"
ragas_path = output_dir / "ragas_acknowledgement_scores.jsonl"

summary_path.write_text(summary.reset_index().to_json(orient="records", indent=2))
scores_path.write_text("\n".join(json.dumps(row, default=str) for row in scores.to_dict(orient="records")) + "\n")
failures_path.write_text("\n".join(json.dumps(row, default=str) for row in failures.to_dict(orient="records")) + "\n")
if "ragas_scores" in globals() and not ragas_scores.empty:
    ragas_path.write_text("\n".join(json.dumps(row, default=str) for row in ragas_scores.to_dict(orient="records")) + "\n")

print(f"Wrote {summary_path}")
print(f"Wrote {scores_path}")
print(f"Wrote {failures_path}")
if "ragas_scores" in globals() and not ragas_scores.empty:
    print(f"Wrote {ragas_path}")

## Presentation Notes Template

Use the measured output above to fill this in:

- **v1 baseline:** basic prompt, weaker explicit definitions, fewer safety/routing instructions.
- **v1 failure modes:** use the failure table, especially missed vulnerability/regulatory signals and unsafe acknowledgement issues.
- **v2 changes:** stronger schema expectations, explicit metadata extraction, explicit vulnerability/regulatory labels, acknowledgement constraints, deterministic routing guardrails.
- **Measured result:** use `summary` and `pivot` above.
- **Next iteration:** expand labelled cases, add human-review rubric for acknowledgement quality, and track false positives for hardship/responsible-lending escalation.